<img src="https://raw.githubusercontent.com/Delarbol/semillero/master/Images-DS/alt_banner.png" width=100%>

<h1 style="margin:0">Taller 4 · NLP y texto como datos</h1>
<h2 style="margin:.35em 0 0 0">Cuaderno asistido para estudiantes</h2>
<p style="margin:.8em 0 0 0"><strong>Duración:</strong> 2 horas · <strong>Modalidad:</strong> Grupos de máximo dos estudiantes</p>
</div>

> **Datos pedagógicos.** Los textos de este cuaderno son simulados. No representan hechos reales y no deben citarse como evidencia económica.

## Identificación de la pareja

- **Estudiante 1:** 
- **Estudiante 2:** 
- **Fecha:** 

Guarden una copia con el formato:

`apellido1_apellido2_taller4_nlp.ipynb`

# Propósito

Este taller está construido con una lógica de **ver–adaptar–explicar**:

1. primero observarán un ejemplo completamente desarrollado;
2. después copiarán y adaptarán ese código a un segundo conjunto de datos;
3. finalmente responderán preguntas que comprueban si entienden la intuición.

La prioridad no es escribir código desde cero. La prioridad es poder explicar **qué hace el código, por qué se usa y qué no permite concluir**.

## Ruta de trabajo sugerida

| Bloque | Tiempo | Actividad |
|---|---:|---|
| 1 | 10 min | Preparación y vocabulario esencial |
| 2 | 25 min | Ejemplo: bolsa de palabras y TF–IDF |
| 3 | 15 min | Ejemplo: embedding y similitud coseno |
| 4 | 25 min | Ejemplo: baseline, pipeline y clasificación |
| 5 | 35 min | Aplicación de la pareja: copiar y adaptar |
| 6 | 10 min | Temas y validación con fuentes |
| **Total** | **120 min** |  |

### Evidencias que deben quedar en el cuaderno

- celdas de código adaptadas y ejecutadas;
- respuestas breves pero justificadas;
- comparación contra un baseline;
- matriz de confusión y análisis de errores;
- interpretación de una representación densa;
- una afirmación contrastada contra fuentes textuales.

# 0. Vocabulario antes de comenzar

| Expresión | Significado sencillo |
|---|---|
| **NLP** | *Natural Language Processing*: procesamiento de lenguaje natural. Técnicas para trabajar computacionalmente con texto. |
| **BoW** | *Bag of Words*: bolsa de palabras. Representa un documento contando términos e ignora su orden. |
| **TF–IDF** | *Term Frequency–Inverse Document Frequency*: frecuencia de término–frecuencia inversa de documento. Da más peso a términos frecuentes en un documento pero poco comunes en el corpus. |
| **SVD** | *Singular Value Decomposition*: descomposición en valores singulares. Reduce una matriz grande a pocas dimensiones densas. |
| **Embedding** | Vector numérico denso que representa un texto en un espacio de menor dimensión. |
| **Baseline** | Regla sencilla que sirve como punto mínimo de comparación. |
| **Pipeline** | Cadena que une transformaciones y modelo para aplicarlas siempre en el orden correcto. |
| **CV** | *Cross-validation*: validación cruzada. Repite entrenamiento y validación en varias particiones de los datos de entrenamiento. |
| **F1 macro** | Promedio del F1 de cada clase dando el mismo peso a todas, incluso a las menos frecuentes. |
| **NMF** | *Non-negative Matrix Factorization*: factorización matricial no negativa. Técnica para descubrir componentes o temas. |

> Cuando aparezca una sigla, regresen a esta tabla. No es necesario memorizar la traducción, pero sí comprender la función.

# 1. Preparación

Ejecuten la siguiente celda. No necesitan modificarla.

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    pairwise_distances,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 110)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42

## Datos para el ejemplo desarrollado

La unidad de análisis es un **fragmento de comunicación económica**. La variable `tema` indica si el fragmento trata principalmente sobre inflación, empleo o actividad económica.

La generación de los textos no es el objetivo del taller. Ejecuten la celda y continúen.

In [ ]:
def crear_corpus_temas(n_por_tema=38, semilla=42):
    rng = np.random.default_rng(semilla)
    temas = {
        'inflación': {
            'sujetos': ['la inflación anual', 'el costo de vida', 'la variación de precios', 'la inflación básica'],
            'objetos': ['alimentos', 'servicios', 'vivienda', 'bienes regulados'],
        },
        'empleo': {
            'sujetos': ['la tasa de desempleo', 'la ocupación urbana', 'el empleo formal', 'la participación laboral'],
            'objetos': ['jóvenes', 'mujeres', 'trabajadores formales', 'zonas urbanas'],
        },
        'actividad': {
            'sujetos': ['la actividad económica', 'la producción industrial', 'el comercio minorista', 'la demanda interna'],
            'objetos': ['industria', 'servicios', 'comercio', 'construcción'],
        },
    }
    movimientos = [
        'aumentó durante el último mes',
        'disminuyó frente al periodo anterior',
        'permaneció relativamente estable',
        'mostró una recuperación gradual',
        'presentó resultados heterogéneos',
    ]
    regiones = ['nacional', 'Caribe', 'Andina', 'Pacífica']
    filas = []
    for tema, elementos in temas.items():
        for i in range(n_por_tema):
            sujeto = rng.choice(elementos['sujetos'])
            objeto = rng.choice(elementos['objetos'])
            texto = f'{sujeto} {rng.choice(movimientos)}. El análisis destacó {objeto}. Cobertura {rng.choice(regiones)}.'
            if rng.random() < 0.50:
                tema_sec = rng.choice([t for t in temas if t != tema])
                sujeto_sec = rng.choice(temas[tema_sec]['sujetos'])
                objeto_sec = rng.choice(temas[tema_sec]['objetos'])
                texto += f' En contraste, también se mencionó {sujeto_sec} y se destacó {objeto_sec}.'
            filas.append({'texto_id': f'EJ-{len(filas)+1:03d}', 'texto': texto, 'tema': tema})
    return pd.DataFrame(filas).sample(frac=1, random_state=semilla).reset_index(drop=True)


df_ejemplo = crear_corpus_temas()
df_ejemplo.head()

## Pregunta inicial

Sin construir un modelo, lean cinco textos al azar y respondan:

1. ¿Qué palabras utilizarían ustedes para reconocer inflación?
2. ¿Qué palabras podrían aparecer en más de un tema?
3. ¿Por qué buscar una sola palabra puede ser insuficiente?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.

In [ ]:
df_ejemplo.sample(5, random_state=7)

# 2. Ejemplo desarrollado: convertir texto en números

## 2.1 Bolsa de palabras

Una bolsa de palabras construye una columna por término y cuenta apariciones.

Ejemplo intuitivo:

- “subió la inflación” → `inflación = 1`, `subió = 1`;
- “bajó la inflación” → `inflación = 1`, `bajó = 1`.

La representación reconoce que ambas frases contienen “inflación”, pero no entiende por sí sola que “subió” y “bajó” expresan direcciones opuestas.

In [ ]:
textos_cortos = pd.Series([
    'La inflación anual bajó por el precio de los alimentos',
    'El empleo formal aumentó en las zonas urbanas',
    'La inflación básica permaneció estable',
], name='texto')

vectorizador_conteo = CountVectorizer(lowercase=True)
matriz_conteo = vectorizador_conteo.fit_transform(textos_cortos)

tabla_conteo = pd.DataFrame(
    matriz_conteo.toarray(),
    columns=vectorizador_conteo.get_feature_names_out(),
    index=['texto 1', 'texto 2', 'texto 3'],
)
tabla_conteo

### Comprueben la intuición

1. ¿Por qué la columna `inflación` vale 1 en dos filas?
2. ¿Qué significa un cero en la tabla?
3. Si escribiéramos “la inflación no bajó”, ¿la bolsa de palabras comprendería correctamente la negación?
4. ¿Cuál es una ventaja de esta representación?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

## 2.2 TF–IDF

**TF–IDF** combina dos ideas:

- **TF — frecuencia de término:** cuánto aparece una palabra en un documento;
- **IDF — frecuencia inversa de documento:** reduce el peso de palabras que aparecen en casi todos los documentos.

Por eso una palabra común como “informe” suele recibir menos peso que una palabra más distintiva como “desempleo”. TF–IDF sigue siendo una representación estadística: no conoce el significado económico de las palabras.

In [ ]:
vectorizador_tfidf_corto = TfidfVectorizer(lowercase=True)
matriz_tfidf_corta = vectorizador_tfidf_corto.fit_transform(textos_cortos)

tabla_tfidf = pd.DataFrame(
    matriz_tfidf_corta.toarray(),
    columns=vectorizador_tfidf_corto.get_feature_names_out(),
    index=['texto 1', 'texto 2', 'texto 3'],
)
tabla_tfidf.round(3)

### Comparación guiada

Comparen `tabla_conteo` y `tabla_tfidf`:

1. ¿Las dos tablas tienen las mismas columnas? ¿Por qué?
2. ¿Por qué los valores TF–IDF no son conteos enteros?
3. ¿Qué representación les parece más útil para clasificación y por qué?
4. ¿TF–IDF puede reconocer que “empleo” y “ocupación” están relacionados semánticamente?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

# 3. Ejemplo desarrollado: embedding y similitud

## 3.1 De muchas columnas a pocas dimensiones

Una matriz TF–IDF puede tener miles de columnas. Aplicaremos **SVD — descomposición en valores singulares** para resumir patrones de coaparición en ocho dimensiones.

El resultado será un **embedding**, es decir, un vector denso para cada texto. “Denso” significa que la mayoría de sus valores no son cero.

In [ ]:
embedding_ejemplo = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('svd', TruncatedSVD(n_components=8, random_state=RANDOM_STATE)),
    ('normalizar', Normalizer(copy=False)),
])

X_embedding_ejemplo = embedding_ejemplo.fit_transform(df_ejemplo['texto'])

print('Número de textos:', X_embedding_ejemplo.shape[0])
print('Dimensiones del embedding:', X_embedding_ejemplo.shape[1])
print('Primer vector:', np.round(X_embedding_ejemplo[0], 3))

## 3.2 Similitud coseno

La similitud coseno compara la dirección de dos vectores:

- valor cercano a 1: los textos son muy cercanos bajo la representación;
- valor cercano a 0: comparten poca señal bajo la representación.

**Importante:** cercanía no significa verdad, causalidad ni respaldo documental.

In [ ]:
def buscar_vecinos(indice_consulta, embeddings, datos, k=5):
    distancias = pairwise_distances(
        embeddings[indice_consulta:indice_consulta + 1],
        embeddings,
        metric='cosine',
    ).ravel()
    orden = np.argsort(distancias)
    orden = orden[orden != indice_consulta][:k]
    salida = datos.loc[orden, ['texto_id', 'tema', 'texto']].copy()
    salida.insert(0, 'similitud', 1 - distancias[orden])
    return salida


indice_consulta = 8
display(df_ejemplo.loc[[indice_consulta], ['texto_id', 'tema', 'texto']])
display(buscar_vecinos(indice_consulta, X_embedding_ejemplo, df_ejemplo, k=5))

### Interpreten, no solo observen

1. ¿Los cinco vecinos pertenecen al mismo tema que la consulta?
2. ¿Qué palabras o expresiones parecen acercarlos?
3. Si un vecino tiene similitud 0,90, ¿podemos afirmar que respalda la consulta?
4. ¿Por qué dos textos que mencionan “aumentó” podrían ser cercanos aunque traten temas diferentes?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

# 4. Ejemplo desarrollado: clasificar textos

Nuestro objetivo será predecir `tema` a partir de `texto`.

## 4.1 Separación entre entrenamiento y prueba

- **Train o entrenamiento:** datos usados para aprender.
- **Test o prueba:** datos aislados para evaluar una sola vez al final.
- **Estratificar:** conservar proporciones similares de las clases en ambas particiones.

Si ajustáramos TF–IDF antes de separar, el vocabulario podría aprender información del test. Eso sería **fuga de información**.

In [ ]:
X_ejemplo = df_ejemplo['texto']
y_ejemplo = df_ejemplo['tema']

X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ejemplo,
    y_ejemplo,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_ejemplo,
)

print('Filas de entrenamiento:', len(X_train_ej))
print('Filas de prueba:', len(X_test_ej))
display(pd.concat([
    y_train_ej.value_counts(normalize=True).rename('train'),
    y_test_ej.value_counts(normalize=True).rename('test'),
], axis=1).round(3))

## 4.2 Baseline

Un baseline responde: “¿qué lograríamos con una regla muy sencilla?”. Usaremos la clase más frecuente.

Si el modelo elaborado no supera claramente esta regla, su complejidad puede no estar justificada.

In [ ]:
baseline_ejemplo = DummyClassifier(strategy='most_frequent')
baseline_ejemplo.fit(X_train_ej.to_frame(), y_train_ej)
pred_base_ej = baseline_ejemplo.predict(X_test_ej.to_frame())

f1_base_ej = f1_score(y_test_ej, pred_base_ej, average='macro')
print('F1 macro del baseline:', round(f1_base_ej, 3))

## 4.3 Pipeline de clasificación

El pipeline tendrá dos pasos:

1. `TfidfVectorizer`: convierte texto en variables numéricas;
2. `LogisticRegression`: aprende a separar las clases.

Aunque se llama regresión logística, aquí se utiliza para **clasificación**.

In [ ]:
modelo_ejemplo = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
    )),
    ('modelo', LogisticRegression(
        max_iter=1200,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    )),
])

cv_ejemplo = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultado_cv_ej = cross_validate(
    modelo_ejemplo,
    X_train_ej,
    y_train_ej,
    cv=cv_ejemplo,
    scoring='f1_macro',
)

print('F1 macro promedio en validación cruzada:', round(resultado_cv_ej['test_score'].mean(), 3))
print('Desviación entre pliegues:', round(resultado_cv_ej['test_score'].std(), 3))

### ¿Qué hizo la validación cruzada?

Dividió únicamente el conjunto de entrenamiento en cinco partes o **pliegues**. En cada repetición entrenó con cuatro y validó con una distinta. El test no participó.

Una media alta con desviación pequeña sugiere estabilidad dentro de estos datos; no garantiza desempeño con textos reales diferentes.

In [ ]:
modelo_ejemplo.fit(X_train_ej, y_train_ej)
pred_ejemplo = modelo_ejemplo.predict(X_test_ej)

f1_modelo_ej = f1_score(y_test_ej, pred_ejemplo, average='macro')
print('F1 macro del baseline:', round(f1_base_ej, 3))
print('F1 macro del modelo:', round(f1_modelo_ej, 3))
print('\nReporte por clase:')
print(classification_report(y_test_ej, pred_ejemplo, digits=3))

## Cómo leer las métricas

- **Precisión de una clase:** entre los textos que el modelo asignó a esa clase, ¿cuántos pertenecían realmente a ella?
- **Recall o sensibilidad:** entre los textos que realmente pertenecían a esa clase, ¿cuántos encontró?
- **F1:** combina precisión y recall.
- **F1 macro:** calcula el F1 por clase y luego les da el mismo peso.

Ejemplo: si había 10 textos reales de empleo y el modelo encontró 8, el recall de empleo sería 8/10 = 0,80.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_ej,
    pred_ejemplo,
    normalize='true',
    cmap='PuRd',
    ax=ax,
)
ax.set_title('Matriz de confusión normalizada por clase real')
plt.tight_layout()
plt.show()

### Lectura de la matriz de confusión

- Las filas representan la clase real.
- Las columnas representan la predicción.
- La diagonal contiene aciertos.
- Los valores fuera de la diagonal muestran confusiones.

Respondan:

1. ¿Qué clase tiene el mayor recall?
2. ¿Qué confusión aparece con mayor frecuencia?
3. ¿El modelo supera el baseline?
4. ¿Podemos concluir que funcionará igual con noticias reales? Expliquen.

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

# 5. Aplicación de la pareja: sentimiento económico

Ahora trabajarán con `df_aplicacion`. La variable objetivo será `tono`:

- `positivo`: el fragmento presenta una evolución favorable;
- `neutral`: describe estabilidad o no toma una dirección clara;
- `negativo`: presenta una evolución desfavorable.

El corpus también tiene `tema_referencia`. **No lo utilicen para predecir tono.** Más adelante servirá únicamente para revisar los temas descubiertos por NMF.

In [ ]:
def crear_corpus_aplicacion(n=180, semilla=2026):
    rng = np.random.default_rng(semilla)
    temas = {
        'precios': ['la inflación', 'el costo de los alimentos', 'los precios de servicios'],
        'empleo': ['la ocupación', 'el desempleo juvenil', 'el empleo formal'],
        'ventas': ['las ventas minoristas', 'la demanda de los hogares', 'el comercio'],
        'crédito': ['la cartera de crédito', 'el acceso al financiamiento', 'la morosidad'],
    }
    frases = {
        'positivo': ['mejoró frente al periodo anterior', 'mostró una recuperación', 'avanzó más de lo previsto'],
        'neutral': ['permaneció estable', 'registró pocos cambios', 'fue descrito sin una conclusión clara'],
        'negativo': ['se deterioró frente al periodo anterior', 'presentó una caída', 'retrocedió más de lo previsto'],
    }
    detalles = ['en zonas urbanas', 'durante el último mes', 'con diferencias regionales', 'según el informe sectorial']
    tonos = np.array(['positivo', 'neutral', 'negativo'])
    probs = np.array([0.30, 0.40, 0.30])
    filas = []
    for i in range(n):
        tema = rng.choice(list(temas))
        tono = rng.choice(tonos, p=probs)
        texto = f"{rng.choice(temas[tema])} {rng.choice(frases[tono])} {rng.choice(detalles)}."
        if rng.random() < 0.35:
            otro_tono = rng.choice([t for t in tonos if t != tono])
            otro_tema = rng.choice([t for t in temas if t != tema])
            texto += f" Sin embargo, {rng.choice(temas[otro_tema])} {rng.choice(frases[otro_tono])}."
        filas.append({
            'texto_id': f'AP-{i+1:03d}',
            'texto': texto,
            'tono': tono,
            'tema_referencia': tema,
        })
    return pd.DataFrame(filas)


df_aplicacion = crear_corpus_aplicacion()
df_aplicacion.head()

## Mapa para adaptar el código

| En el ejemplo | En la aplicación |
|---|---|
| `df_ejemplo` | `df_aplicacion` |
| `tema` | `tono` |
| `X_train_ej` | pueden usar `X_train_ap` |
| `y_train_ej` | pueden usar `y_train_ap` |
| `modelo_ejemplo` | pueden usar `modelo_aplicacion` |

Copiar está permitido y es parte del ejercicio. Sin embargo, deben poder explicar cada cambio.

## Actividad 1 · Auditoría básica — 10 minutos

Copien y adapten comandos usados anteriormente para responder:

1. ¿Cuántas filas hay?
2. ¿Existen textos duplicados?
3. ¿Existen valores faltantes?
4. ¿Cuál es la distribución de `tono`?
5. ¿Cuál sería la clase elegida por un baseline de clase mayoritaria?

**Código útil observado en el ejemplo:** `shape`, `nunique()`, `isna()`, `value_counts()`.

In [ ]:
# ACTIVIDAD 1 — escriban y ejecuten aquí su código adaptado.
# Pista de inicio:
# df_aplicacion.shape

> **Interpretación de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.  
> 5.

## Actividad 2 · Split y baseline — 10 minutos

Copien el bloque de la sección 4.1 y adapten:

- `X` debe ser la columna `texto`;
- `y` debe ser la columna `tono`;
- usen 25 % para test;
- estratifiquen con `y`;
- conserven `RANDOM_STATE`.

Después copien el baseline de la sección 4.2 y calculen F1 macro.

In [ ]:
# ACTIVIDAD 2 — escriban aquí X_aplicacion, y_aplicacion y el split.
# Luego ajusten un DummyClassifier con strategy='most_frequent'.

### Preguntas de comprensión

1. ¿Por qué `stratify=y_aplicacion` es útil?
2. ¿Por qué el test no debe utilizarse para escoger el modelo?
3. ¿Por qué un baseline de clase mayoritaria puede tener accuracy razonable y F1 macro bajo?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.

## Actividad 3 · Pipeline y validación cruzada — 15 minutos

Copien el pipeline completo del ejemplo. Cambien solamente los nombres de las variables.

Su resultado debe incluir:

- TF–IDF con unigramas y bigramas;
- regresión logística balanceada;
- validación cruzada estratificada de cinco pliegues;
- media y desviación de F1 macro.

In [ ]:
# ACTIVIDAD 3 — copien y adapten modelo_ejemplo y resultado_cv_ej.
# Nombres sugeridos: modelo_aplicacion y resultado_cv_ap.

### Preguntas de comprensión

1. ¿Qué aprende TF–IDF y qué aprende la regresión logística?
2. ¿Por qué ambos pasos deben vivir en el mismo pipeline?
3. ¿Qué significaría una desviación grande entre los cinco pliegues?
4. Si F1 macro es 0,80, ¿significa que el modelo tiene 80 % de probabilidad de decir la verdad?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

## Actividad 4 · Evaluación final — 10 minutos

Copien las celdas que:

1. ajustan el modelo con `fit`;
2. producen predicciones con `predict`;
3. calculan F1 macro;
4. muestran `classification_report`;
5. construyen la matriz de confusión normalizada.

No ajusten nuevamente hiperparámetros después de mirar el test.

In [ ]:
# ACTIVIDAD 4 — evaluación final en test.

### Lectura obligatoria del resultado

1. Comparen F1 macro del modelo y del baseline. ¿Cuál es la mejora?
2. ¿Qué clase tiene menor recall?
3. Identifiquen una confusión frecuente y lean tres textos involucrados.
4. ¿El error parece provenir del modelo o de que el texto es genuinamente ambiguo?
5. ¿Qué decisión **no** debería automatizarse con este modelo?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.  
> 5.

## Actividad 5 · Embedding y vecinos — 10 minutos

Copien y adapten el código de la sección 3:

- ajusten TF–IDF + SVD sobre `df_aplicacion['texto']`;
- usen 8 dimensiones;
- elijan un texto de consulta;
- recuperen cinco vecinos mediante similitud coseno.

Pueden reutilizar la función `buscar_vecinos`, pero deberán cambiar la columna de etiqueta que muestra.

In [ ]:
# ACTIVIDAD 5 — embedding y vecinos del corpus de aplicación.
# Pista: creen una copia de buscar_vecinos que muestre 'tono' y 'tema_referencia'.

### Preguntas de comprensión

1. ¿Los vecinos comparten tono, tema, vocabulario o una combinación?
2. ¿Encontraron algún vecino cercano con tono diferente?
3. ¿Por qué la similitud coseno no es una métrica de veracidad?
4. ¿Qué información se pierde al reducir miles de columnas a ocho dimensiones?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

# 6. Temas no supervisados

La clasificación anterior era supervisada porque conocíamos `tono`. Ahora usaremos **NMF — factorización matricial no negativa** para encontrar componentes sin darle al algoritmo `tema_referencia`.

NMF entrega grupos de palabras. El algoritmo no entrega nombres como “empleo” o “precios”; esos nombres los propone una persona después de leer palabras y documentos representativos.

## Ejemplo desarrollado con el primer corpus

El código encuentra tres componentes porque sabemos que el ejemplo fue construido alrededor de tres temas. En un problema real, el número de componentes sería una decisión que también debe validarse.

Retiraremos palabras genéricas de la plantilla, como “último mes” o “resultados”, para que no dominen los componentes. Esta lista debe documentarse: eliminar palabras sin justificación puede borrar información importante.

In [ ]:
stopwords_nmf = [
    'la', 'el', 'los', 'las', 'un', 'una', 'de', 'del', 'en', 'y', 'se',
    'durante', 'último', 'mes', 'periodo', 'anterior', 'análisis', 'destacó',
    'cobertura', 'también', 'mencionó', 'contraste', 'presentó', 'resultados',
    'heterogéneos', 'mostró', 'recuperación', 'gradual', 'aumentó', 'disminuyó',
    'permaneció', 'relativamente', 'estable', 'frente',
]

tfidf_nmf_ej = TfidfVectorizer(
    ngram_range=(1, 2), min_df=3, max_df=0.90, stop_words=stopwords_nmf
)
X_nmf_ej = tfidf_nmf_ej.fit_transform(df_ejemplo['texto'])

nmf_ejemplo = NMF(n_components=3, init='nndsvda', random_state=RANDOM_STATE, max_iter=500)
W_ejemplo = nmf_ejemplo.fit_transform(X_nmf_ej)
H_ejemplo = nmf_ejemplo.components_
vocab_nmf_ej = tfidf_nmf_ej.get_feature_names_out()

for componente, pesos in enumerate(H_ejemplo):
    top = np.argsort(pesos)[-8:][::-1]
    print(f'Componente {componente}:', ', '.join(vocab_nmf_ej[top]))

## Actividad 6 · Copiar y adaptar NMF — 10 minutos

Adapten el ejemplo a `df_aplicacion`:

1. utilicen solo `texto` para ajustar NMF;
2. soliciten cuatro componentes;
3. impriman ocho palabras por componente;
4. propongan un nombre para cada componente;
5. solo después comparen el componente dominante con `tema_referencia`.

**No miren primero `tema_referencia` para bautizar los componentes.**

In [ ]:
# ACTIVIDAD 6 — copien el bloque NMF y adapten:
# df_ejemplo -> df_aplicacion
# n_components=3 -> n_components=4

| Componente | Nombre propuesto | Palabras que lo justifican | Ambigüedad encontrada |
|---:|---|---|---|
| 0 |  |  |  |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

### Preguntas

1. ¿Un componente equivale necesariamente a un tema verdadero?
2. ¿Qué componente fue más fácil de nombrar?
3. ¿Cuál mezcló vocabulario de varios temas?
4. ¿Por qué NMF no debe evaluarse como si sus números fueran etiquetas ya conocidas?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.

# 7. Validación con fuentes textuales

Una similitud alta sirve para **recuperar candidatos**. La validación exige leer la fuente y revisar:

- variable;
- dirección del cambio;
- periodo;
- población o región;
- lenguaje causal;
- posibles contradicciones.

Primero veamos un ejemplo completo.

In [ ]:
fuentes = pd.DataFrame([
    ['F-01', 'La inflación anual disminuyó. La reducción se concentró en alimentos, mientras los servicios mostraron persistencia.'],
    ['F-02', 'La ocupación formal aumentó en zonas urbanas, pero el informe no identifica el efecto causal de una política.'],
    ['F-03', 'Las ventas minoristas permanecieron estables y presentaron diferencias importantes entre regiones.'],
    ['F-04', 'La cartera de crédito creció, aunque la morosidad también aumentó durante el trimestre.'],
    ['F-05', 'Las cifras regionales no son comparables sin revisar cobertura, población y cambios metodológicos.'],
], columns=['fuente_id', 'fragmento'])


def recuperar_evidencia(afirmacion, fuentes_df=fuentes, k=2):
    documentos = fuentes_df['fragmento'].tolist() + [afirmacion]
    vectorizador = TfidfVectorizer(ngram_range=(1, 2))
    matriz = vectorizador.fit_transform(documentos)
    similitudes = 1 - pairwise_distances(matriz[-1], matriz[:-1], metric='cosine').ravel()
    idx = np.argsort(similitudes)[-k:][::-1]
    salida = fuentes_df.iloc[idx].copy()
    salida.insert(0, 'similitud', similitudes[idx])
    return salida.reset_index(drop=True)


afirmacion_ejemplo = 'Una política pública causó el aumento de la ocupación formal.'
print('Afirmación:', afirmacion_ejemplo)
recuperar_evidencia(afirmacion_ejemplo)

## Lectura del ejemplo

F-02 es textualmente cercana porque menciona ocupación formal y política. Sin embargo, dice expresamente que el informe **no identifica causalidad**. Por lo tanto, recuperar una fuente cercana no confirma la afirmación: en este caso revela que su alcance es injustificado.

## Actividad 7 · Una afirmación y su evidencia — 10 minutos

Redacten una afirmación breve a partir de sus resultados. Después:

1. usen `recuperar_evidencia`;
2. lean las dos fuentes candidatas;
3. clasifiquen la afirmación como `respaldada`, `parcial`, `contradicha` o `sin evidencia suficiente`;
4. reescriban la afirmación para que coincida con la evidencia disponible.

In [ ]:
# ACTIVIDAD 7 — reemplacen el texto y ejecuten.
afirmacion_pareja = 'REEMPLAZAR POR LA AFIRMACIÓN DE LA PAREJA'

if afirmacion_pareja.startswith('REEMPLAZAR'):
    print('Escriban primero su afirmación.')
else:
    display(recuperar_evidencia(afirmacion_pareja, k=2))

- **Afirmación original:** 
- **Fuentes revisadas:** 
- **Estado:** 
- **Justificación:** 
- **Afirmación corregida:** 

### Pregunta final de esta sección

¿Qué trabajo realiza la similitud coseno y qué trabajo debe seguir realizando una persona?

> **Respuesta de la pareja:**

# 8. Cierre: cinco ideas que deben poder explicar

Respondan sin copiar definiciones textuales:

1. ¿En qué se diferencian BoW, TF–IDF y un embedding?
2. ¿Por qué el vectorizador debe ajustarse dentro del pipeline?
3. ¿Qué demuestra que un modelo supera al baseline?
4. ¿Qué información aporta una matriz de confusión que no aporta un único F1 macro?
5. ¿Por qué una fuente similar no es necesariamente una fuente que respalda?

> **Respuesta de la pareja:**  
> 1.  
> 2.  
> 3.  
> 4.  
> 5.

## Autoevaluación antes de entregar

Marquen cada punto:

- [ ] El cuaderno identifica a los dos integrantes.
- [ ] Todas las celdas de código fueron ejecutadas en orden.
- [ ] Comparamos el modelo con un baseline.
- [ ] No usamos el test para escoger el modelo.
- [ ] Interpretamos F1 macro y una matriz de confusión.
- [ ] Recuperamos vecinos en un embedding y evitamos llamarlos “prueba”.
- [ ] Nombramos componentes NMF leyendo palabras y documentos.
- [ ] Validamos una afirmación contra fragmentos fuente.
- [ ] Respondimos con nuestras propias palabras.
- [ ] Reconocimos que los datos son simulados.

> El código puede copiarse y adaptarse. Las interpretaciones deben pertenecer a la pareja.